# 16S Analyses of the Longitudinal Acne Study
## Random Forest analysis

Date created: 12/21/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, and Britta De Pessemier

This notebook plots the following:

- Random forest analysis showing the AUCs for separating the three groups for each primer pair and the driving taxa.

In [199]:
# Import Python packages
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils import resample
import numpy as np
import pandas as pd
import qiime2
import matplotlib.pyplot as plt
from biom import load_table
from matplotlib_venn import venn3, venn3_circles
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from matplotlib.transforms import Affine2D
import seaborn as sns
from upsetplot import from_memberships, plot
from upsetplot import UpSet, from_contents
import os

In [200]:
def qza_to_dataframe(qza_path, metadata_path, column):
    """
    Converts a QIIME 2 .qza table to a pandas DataFrame and appends a column of interest from metadata.

    Parameters:
    qza_path (str): Path to the QIIME 2 .qza file.
    metadata_path (str): Path to the metadata CSV file.
    column (str): Name of the column in the metadata to be added as the label.

    Returns:
    pd.DataFrame: A DataFrame with features as rows, samples as columns, and the category label as the last column.
    """
    import qiime2
    import pandas as pd

    # Load the QIIME 2 Artifact
    table_artifact = qiime2.Artifact.load(qza_path)

    # Export the BIOM table to a pandas DataFrame
    biom_table = table_artifact.view(pd.DataFrame)

    # Load metadata
    metadata = pd.read_csv(metadata_path, index_col=0)  # Assuming sample IDs are in the index

    # Add the category label column from metadata
    biom_table[column] = metadata[column]

    # Remove rows with NaN values in the specified column
    biom_table = biom_table.dropna(subset=[column])

    return biom_table


# Load data for V1-V3 and V4
v1_v3_data = qza_to_dataframe('../Data/16S/Tables/16S_V1-V3_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')
v4_data = qza_to_dataframe('../Data/16S/Tables/16S_V4_rarefied_uncollapsed_table.qza', '../Metadata/57610_57610_analysis_mapping_category_label.csv', 'category_label')

In [201]:
def random_forest_auc_binary(features_df, n_estimators_list, label_column, categories):
    """
    Perform Random Forest classification and calculate AUC for two categories.

    Parameters:
    features_df (pd.DataFrame): DataFrame with features and a label column.
    n_estimators_list (list): List of numbers of trees for Random Forest.
    label_column (str): Name of the column containing class labels.
    categories (list): Two categories to filter and analyze.

    Returns:
    dict: AUC scores for each number of trees.
    """

    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]

    # Extract features and labels
    labels = filtered_df[label_column]
    features = filtered_df.drop(columns=[label_column])

    # Map the two categories to binary labels (e.g., 0 and 1)
    labels_binary = labels.map({categories[0]: 0, categories[1]: 1})

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels_binary, stratify=labels_binary, test_size=0.2, random_state=42
    )

    auc_scores = {}
    for n_trees in n_estimators_list:
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
        rf.fit(X_train, y_train)

        # Predict probabilities
        y_prob = rf.predict_proba(X_test)[:, 1]  # Get probabilities for the positive class

        # Calculate AUC
        auc = roc_auc_score(y_test, y_prob)
        auc_scores[n_trees] = auc

    return auc_scores


In [202]:
# Run Random Forest Analysis between Acne Lesional and Healthy
categories = ["Acne Lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Healthy):", v4_auc)

V1-V3 AUC scores (Acne Lesional vs. Healthy): {100: 0.9846743295019157, 500: 0.9885057471264368, 1000: 0.996168582375479}
V4 AUC scores (Acne Lesional vs. Healthy): {100: 0.8500000000000001, 500: 0.8466666666666667, 1000: 0.8566666666666667}


In [203]:
# Run Random Forest Analysis between Acne Non-lesional and Healthy
categories = ["Acne Non-lesional", "Healthy"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Non-lesional vs. Healthy):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Non-lesional vs. Healthy):", v4_auc)

V1-V3 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.9611111111111111, 500: 0.9555555555555555, 1000: 0.9444444444444444}
V4 AUC scores (Acne Non-lesional vs. Healthy): {100: 0.851851851851852, 500: 0.8703703703703703, 1000: 0.8703703703703703}


In [204]:
# Run Random Forest Analysis between Acne Lesional and Acne Non-lesional
categories = ["Acne Lesional", "Acne Non-lesional"]
n_estimators_list = [100, 500, 1000]
label_column = "category_label"

# Perform analysis for V1-V3
v1_v3_auc = random_forest_auc_binary(v1_v3_data, n_estimators_list, label_column, categories)
print("V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional):", v1_v3_auc)

# Perform analysis for V4
v4_auc = random_forest_auc_binary(v4_data, n_estimators_list, label_column, categories)
print("V4 AUC scores (Acne Lesional vs. Acne Non-lesional):", v4_auc)

V1-V3 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.6551724137931035, 500: 0.6819923371647509, 1000: 0.6781609195402298}
V4 AUC scores (Acne Lesional vs. Acne Non-lesional): {100: 0.49444444444444435, 500: 0.4851851851851852, 1000: 0.4666666666666667}


## ROC plot

In [205]:
def plot_roc_curves_with_ci(features_df, label_column, categories, n_estimators, primer_set_label, color, linestyle='-'):
    """
    Generate ROC curves with a fixed confidence interval (±0.05) for a specific primer set and category comparison.

    Parameters:
    features_df (pd.DataFrame): DataFrame containing features and labels.
    label_column (str): Column name containing the labels.
    categories (list): Two categories to filter and compare.
    n_estimators (int): Number of trees for Random Forest.
    primer_set_label (str): Label for the primer set (e.g., "V1-V3").
    color (str): Color for the ROC curve and confidence interval.
    linestyle (str): Line style for the ROC curve (default is solid line '-').
    """
    # Filter DataFrame for the specified categories
    filtered_df = features_df[features_df[label_column].isin(categories)]
    labels = filtered_df[label_column].map({categories[0]: 0, categories[1]: 1})
    features = filtered_df.drop(columns=[label_column])

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, stratify=labels, test_size=0.2, random_state=42
    )

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    rf.fit(X_train, y_train)

    # Predict probabilities
    y_prob = rf.predict_proba(X_test)[:, 1]

    # Calculate ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    # Define fixed confidence interval
    ci_margin = 0.05
    lower_tpr = np.maximum(tpr - ci_margin, 0)
    upper_tpr = np.minimum(tpr + ci_margin, 1)

    # Plot the ROC curve with fixed confidence intervals
    plt.plot(fpr, tpr, label=f"{primer_set_label} (AUC = {roc_auc:.2f} ± 0.05)", color=color, linestyle=linestyle, linewidth=2.5)
    plt.fill_between(fpr, lower_tpr, upper_tpr, color=color, alpha=0.2)


# Categories for comparisons
categories_lesional_vs_healthy = ["Acne Lesional", "Healthy"]
categories_lesional_vs_non_lesional = ["Acne Lesional", "Acne Non-lesional"]
categories_non_lesional_vs_healthy = ["Acne Non-lesional", "Healthy"]

# Number of trees for Random Forest
n_estimators = 1000

# Create ROC plot
plt.figure(figsize=(9, 8))

# V4 data (dotted lines)
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V4 AL vs ANL", "#f16c52", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V4 H vs ANL", "#5cbccb", linestyle="--")
plot_roc_curves_with_ci(v4_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V4 H vs AL)", "#3333B3", linestyle="--")

# V1-V3 data (solid lines)
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_non_lesional, n_estimators, "V1-V3 AL vs ANL", "#f16c52")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_non_lesional_vs_healthy, n_estimators, "V1-V3 H vs ANL", "#5cbccb")
plot_roc_curves_with_ci(v1_v3_data, "category_label", categories_lesional_vs_healthy, n_estimators, "V1-V3 H vs AL", "#3333B3")

# Customize plot
plt.plot([0, 1], [0, 1], color="black", linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate", fontsize=18)
plt.ylabel("True Positive Rate", fontsize=18)
plt.title("V1-V3 vs V4 Random Forest (RF) ROC Curves", fontsize=20)

# Define the desired legend order
desired_order = [
    "V1-V3 H vs AL (AUC = 1.00 ± 0.05)",
    "V4 H vs AL) (AUC = 0.86 ± 0.05)",
    "V1-V3 H vs ANL (AUC = 0.94 ± 0.05)",
    "V4 H vs ANL (AUC = 0.87 ± 0.05)",
    "V1-V3 AL vs ANL (AUC = 0.68 ± 0.05)",
    "V4 AL vs ANL (AUC = 0.47 ± 0.05)"
]

# Get current handles and labels from the legend
handles, labels = plt.gca().get_legend_handles_labels()

# Reorder handles and labels based on the desired order
sorted_handles = []
sorted_labels = []
for label in desired_order:
    if label in labels:
        index = labels.index(label)
        sorted_handles.append(handles[index])
        sorted_labels.append(labels[index])

# Apply the reordered legend
plt.legend(sorted_handles, sorted_labels, loc="lower right", fontsize=12)

# Finalize and show the plot
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.grid()
plt.tight_layout()
plt.savefig('../Figures/16S_Figures/primer_comparison/roc_curves_with_ci.png', dpi=600)


## V1-V3 taxa driving importance by skin group

In [206]:
# Load the BIOM table
biom_file = "../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom"
table = load_table(biom_file)

biom_df = pd.DataFrame(table.matrix_data.toarray(), 
                       index=table.ids(axis='observation'), 
                       columns=table.ids(axis='sample'))
biom_df = biom_df.T
biom_df

,g__Acinetobacter,g__Actinomyces,g__Alloprevotella,g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium,g__Anaerococcus,g__Auricoccus-Abyssicoccus,g__Bergeyella,g__Blastomonas,g__Brevibacterium,g__Brevundimonas,...,g__Sediminibacterium,g__Sphingopyxis,g__Sphingorhabdus,g__Staphylococcus,g__Streptococcus,g__Subdoligranulum,g__Thermus,g__Veillonella,g__Xanthomonas,g__uncultured
LAMI.RD001.D0.C1,0.0,0.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,94.0,292.0,0.0,6.0,33.0,0.0,0.0
LAMI.RD001.D0.C2,0.0,1.0,27.0,0.0,4.0,0.0,0.0,0.0,6.0,2.0,...,0.0,0.0,0.0,158.0,374.0,0.0,1.0,36.0,0.0,2.0
LAMI.RD001.D14.C1,0.0,3.0,30.0,0.0,0.0,0.0,1.0,2.0,95.0,0.0,...,0.0,0.0,1.0,83.0,240.0,0.0,0.0,19.0,0.0,2.0
LAMI.RD001.D14.C2,0.0,8.0,100.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,...,0.0,0.0,0.0,137.0,446.0,0.0,0.0,69.0,0.0,0.0
LAMI.RD001.D28.C1,0.0,1.0,38.0,0.0,0.0,0.0,0.0,0.0,14.0,1.0,...,2.0,0.0,1.0,118.0,293.0,0.0,0.0,34.0,0.0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D14.C1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,1846.0
LAMI.RD319.D21.C3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,11.0,1.0,0.0,0.0,0.0,0.0,2741.0
LAMI.RD319.D14.C2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,1410.0
LAMI.RD319.D9.C1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,13.0,1.0,0.0,0.0,0.0,0.0,2491.0


In [207]:
# Group samples by metadata
metadata = pd.read_csv('../Metadata/V1V3_metadata_group.tsv', index_col=None, sep='\t')
metadata

,SampleID,group
0,LAMI.RD308.D16.C1,Acne_L
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
...,...,...
231,LAMI.RD317.D21.C1,Acne_L
232,LAMI.RD001.D0.C1,Healthy
233,LAMI.RD014.D14.C2,Healthy
234,LAMI.RD314.D0.C1,Acne_L


In [208]:
# Assuming 'sample_id' links the two DataFrames
merged_df = biom_df.merge(metadata[['SampleID', 'group']], left_index=True, right_on='SampleID')

# Remove merged_df column
merged_df.drop(columns=['SampleID'], inplace=True)

merged_df

,g__Acinetobacter,g__Actinomyces,g__Alloprevotella,g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium,g__Anaerococcus,g__Auricoccus-Abyssicoccus,g__Bergeyella,g__Blastomonas,g__Brevibacterium,g__Brevundimonas,...,g__Sphingopyxis,g__Sphingorhabdus,g__Staphylococcus,g__Streptococcus,g__Subdoligranulum,g__Thermus,g__Veillonella,g__Xanthomonas,g__uncultured,group
232,0.0,0.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,94.0,292.0,0.0,6.0,33.0,0.0,0.0,Healthy
109,0.0,1.0,27.0,0.0,4.0,0.0,0.0,0.0,6.0,2.0,...,0.0,0.0,158.0,374.0,0.0,1.0,36.0,0.0,2.0,Healthy
33,0.0,3.0,30.0,0.0,0.0,0.0,1.0,2.0,95.0,0.0,...,0.0,1.0,83.0,240.0,0.0,0.0,19.0,0.0,2.0,Healthy
235,0.0,8.0,100.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,...,0.0,0.0,137.0,446.0,0.0,0.0,69.0,0.0,0.0,Healthy
40,0.0,1.0,38.0,0.0,0.0,0.0,0.0,0.0,14.0,1.0,...,0.0,1.0,118.0,293.0,0.0,0.0,34.0,0.0,11.0,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,1846.0,Acne_L
150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,11.0,1.0,0.0,0.0,0.0,0.0,2741.0,Acne_NL
158,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,1410.0,Acne_L
189,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,13.0,1.0,0.0,0.0,0.0,0.0,2491.0,Acne_L


In [209]:
def one_vs_all_gini_importance(features_df, label_column, groups, n_trees=1000):
    """
    Compute Gini importance for each group in a one-vs-all classification.

    Parameters:
    features_df (pd.DataFrame): DataFrame with features and a label column.
    label_column (str): Name of the column containing class labels.
    groups (list): List of all unique groups in the label column.
    n_trees (int): Number of trees in the Random Forest.

    Returns:
    dict: Feature importance DataFrames for each group.
    """
    results = {}
    
    for group in groups:
        # Create binary labels for one-vs-all classification
        binary_labels = features_df[label_column].apply(lambda x: 1 if x == group else 0)
        
        # Extract features
        features = features_df.drop(columns=[label_column])
        
        # Train Random Forest
        rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
        rf.fit(features, binary_labels)
        
        # Get feature importances
        importances = rf.feature_importances_
        
        # Create a DataFrame of feature importances
        importance_df = pd.DataFrame({
            "Feature": features.columns,
            "Importance": importances
        }).sort_values(by="Importance", ascending=False)
        
        results[group] = importance_df
    
    return results

# Define your groups
groups = merged_df["group"].unique()

# Perform one-vs-all Gini importance analysis
one_vs_all_results_V1V3 = one_vs_all_gini_importance(merged_df, "group", groups)


In [210]:
def save_one_vs_all_results(results, output_path):
    """
    Save one-vs-all feature importance results to a CSV file.

    Parameters:
    results (dict): Dictionary where keys are group names, and values are DataFrames with feature importances.
    output_path (str): Path to save the combined CSV file.
    """
    # Combine all results into a single DataFrame with group identifiers
    combined_results = pd.concat(
        [df.assign(Group=group) for group, df in results.items()],
        ignore_index=True
    )
    
    # Save to CSV
    combined_results.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")

# Save the one_vs_all_results to a CSV
save_one_vs_all_results(one_vs_all_results_V1V3, '../Data/16S/random_forest/V1V3/one_vs_all_results_V1V3.csv')

OSError: Cannot save file into a non-existent directory: '../Data/16S/random_forest/V1V3'

In [ ]:
# Load the results CSV file
results_df = pd.read_csv('../Data/16S/random_forest/V1V3/one_vs_all_results_V1V3.csv')

# Specify the taxa of interest
taxa_of_interest = [
    'g__Cutibacterium', 'g__uncultured', 'g__Staphylococcus', 'g__Streptococcus', 'g__Corynebacterium', 
    'g__Lawsonella', 'g__Micrococcus', 'g__Alloprevotella', 'g__Lactobacillus', 'g__Rothia'
    #'g__Neisseria', 'g__Porphyromonas', 'g__Prevotella',  'g__Gemella', 'g__Veillonella'
]

# Filter the data for the specified taxa
filtered_df = results_df[results_df['Feature'].str.strip().isin(taxa_of_interest)]

# Ensure the x-axis order is Healthy, Acne Non-lesional, Acne Lesional
group_order = ["Healthy", "Acne_NL", "Acne_L"]
filtered_df["Group"] = pd.Categorical(filtered_df["Group"], categories=group_order, ordered=True)

# Define custom colors for the taxa
taxa_colors = {
    'g__Cutibacterium': '#ffa505',  # Bright orange
    'g__Staphylococcus': '#92f0f0',  # Fluorescent light blue
    'g__Streptococcus': '#e2b46c',  # Beige
    'g__Corynebacterium': '#ffe59a',  # Pastel yellow
    'g__Lawsonella': '#70a8dc',  # Light blue
    'g__Veillonella': '#c5bce0',  # Pastel purplish
    'g__Micrococcus': '#f4cccd',  # Pastel yellow
    'g__Alloprevotella': '#bcbcbc',  # Light gray
    'g__Lactobacillus': '#daead3',  # Pale mint green
    'g__Rothia': '#ccb974', # Muted olive green
    'g__uncultured': '#f6475f',  # Pinkish
    'g__Neisseria': '#f6475f',  # Pinkish
    'g__Porphyromonas': '#64b5cd', # Light sky blue
    'g__Prevotella': '#4c72b0', # Muted royal blue 
    'g__Gemella': '#dd8452', # Light brown

}

# Strip leading/trailing spaces from 'Feature' column for consistency
filtered_df['Feature'] = filtered_df['Feature'].str.strip()

# Match the palette with the features in the filtered DataFrame
custom_palette = {feature: taxa_colors[feature] for feature in filtered_df['Feature'].unique()}

# Create the line plot
plt.figure(figsize=(7, 8))
sns.lineplot(
    data=filtered_df,
    x="Group",
    y="Importance",
    hue="Feature",
    marker="o",
    palette=custom_palette,
    hue_order=taxa_of_interest  # Explicitly set the order of taxa in the legend
)

# Add labels and title
plt.title("RF Driving Taxa by 16S V1-V3", fontsize=20)
plt.xlabel("")
plt.ylabel("Feature Importance (Gini)", fontsize=18)
plt.xticks(ticks=[0, 1, 2], labels=["H", "ANL", "AL"], fontsize=18)
plt.yticks(fontsize=14)

# Create custom legend with circles in the middle of the lines
legend_elements = [
    Line2D(
        [0], [0],
        color=taxa_colors[taxon], marker='o', markersize=5, label=taxon,
        linestyle='-', linewidth=2
    )
    for taxon in taxa_of_interest if taxon in custom_palette
]

# Add the custom legend
plt.legend(
    handles=legend_elements,
    title="Top Genera",
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    title_fontsize=14,
    fontsize=14
)

# Save and show the plot
plt.tight_layout()
plt.savefig('../Figures/16S_Figures/primer_comparison/V1V3_Gini_importance.png', dpi=600)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_76434/2544437351.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["Group"] = pd.Categorical(filtered_df["Group"], categories=group_order, ordered=True)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_76434/2544437351.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Feature'] = filtered_df['Feature'].str.strip()
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_

## V4 taxa driving importance by group

In [ ]:
# Load the BIOM table
biom_file = "../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom"
table = load_table(biom_file)

biom_df = pd.DataFrame(table.matrix_data.toarray(), 
                       index=table.ids(axis='observation'), 
                       columns=table.ids(axis='sample'))
biom_df = biom_df.T
biom_df

,g__Abiotrophia,g__Absconditabacteriales_(SR1),g__Acinetobacter,g__Actinomyces,g__Aerococcus,g__Aeromonas,g__Agathobacter,g__Aggregatibacter,g__Alloiococcus,g__Alloprevotella,...,g__Stenotrophomonas,g__Streptococcus,g__Thermus,g__Treponema,g__Turicella,g__Veillonella,g__Vibrio,g__Weissella,g__Xanthomonas,g__uncultured
LAMI.RD001.D0.C1,14.0,38.0,116.0,19.0,9.0,0.0,0.0,160.0,0.0,18.0,...,0.0,414.0,0.0,0.0,46.0,43.0,0.0,0.0,0.0,61.0
LAMI.RD001.D14.C1,21.0,0.0,57.0,48.0,3.0,2.0,2.0,23.0,0.0,26.0,...,0.0,247.0,0.0,0.0,11.0,41.0,1.0,0.0,1.0,29.0
LAMI.RD004.D0.C2,0.0,0.0,72.0,2.0,63.0,3.0,0.0,0.0,0.0,0.0,...,0.0,164.0,0.0,0.0,0.0,59.0,0.0,0.0,0.0,36.0
LAMI.RD001.D0.C2,9.0,19.0,83.0,10.0,4.0,0.0,0.0,61.0,0.0,66.0,...,0.0,368.0,5.0,0.0,0.0,37.0,0.0,0.0,0.0,46.0
LAMI.RD004.D28.C1,0.0,0.0,326.0,6.0,32.0,65.0,0.0,0.0,0.0,16.0,...,17.0,422.0,0.0,2.0,0.0,35.0,0.0,0.0,1.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D16.C2,0.0,0.0,3.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,3413.0
LAMI.RD319.D28.C1,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,7.0,0.0,0.0,16.0,1.0,0.0,0.0,0.0,3525.0
LAMI.RD318.D9.C2,0.0,0.0,35.0,0.0,2.0,0.0,0.0,0.0,4.0,1.0,...,0.0,11.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3441.0
LAMI.RD319.D28.C2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,3627.0


In [ ]:
# Group samples by metadata
metadata = pd.read_csv('../Metadata/V4_metadata_group.tsv', index_col=None, sep='\t')
metadata

,SampleID,group
0,LAMI.RD308.D16.C1,Acne_L
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
4,LAMI.RD306.D7.C2,Acne_L
...,...,...
212,LAMI.RD305.D0.C2,Acne_L
213,LAMI.RD317.D21.C1,Acne_L
214,LAMI.RD001.D0.C1,Healthy
215,LAMI.RD314.D0.C1,Acne_L


In [ ]:
# Assuming 'sample_id' links the two DataFrames
merged_df = biom_df.merge(metadata[['SampleID', 'group']], left_index=True, right_on='SampleID')

# Remove merged_df column
merged_df.drop(columns=['SampleID'], inplace=True)

merged_df

,g__Abiotrophia,g__Absconditabacteriales_(SR1),g__Acinetobacter,g__Actinomyces,g__Aerococcus,g__Aeromonas,g__Agathobacter,g__Aggregatibacter,g__Alloiococcus,g__Alloprevotella,...,g__Streptococcus,g__Thermus,g__Treponema,g__Turicella,g__Veillonella,g__Vibrio,g__Weissella,g__Xanthomonas,g__uncultured,group
214,14.0,38.0,116.0,19.0,9.0,0.0,0.0,160.0,0.0,18.0,...,414.0,0.0,0.0,46.0,43.0,0.0,0.0,0.0,61.0,Healthy
29,21.0,0.0,57.0,48.0,3.0,2.0,2.0,23.0,0.0,26.0,...,247.0,0.0,0.0,11.0,41.0,1.0,0.0,1.0,29.0,Healthy
196,0.0,0.0,72.0,2.0,63.0,3.0,0.0,0.0,0.0,0.0,...,164.0,0.0,0.0,0.0,59.0,0.0,0.0,0.0,36.0,Healthy
102,9.0,19.0,83.0,10.0,4.0,0.0,0.0,61.0,0.0,66.0,...,368.0,5.0,0.0,0.0,37.0,0.0,0.0,0.0,46.0,Healthy
40,0.0,0.0,326.0,6.0,32.0,65.0,0.0,0.0,0.0,16.0,...,422.0,0.0,2.0,0.0,35.0,0.0,0.0,1.0,16.0,Healthy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,0.0,0.0,3.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,2.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,3413.0,Acne_L
39,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,0.0,0.0,16.0,1.0,0.0,0.0,0.0,3525.0,Acne_L
83,0.0,0.0,35.0,0.0,2.0,0.0,0.0,0.0,4.0,1.0,...,11.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3441.0,Acne_L
202,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,3627.0,Acne_L


In [ ]:
# Perform one-vs-all Gini importance analysis
one_vs_all_results_V4 = one_vs_all_gini_importance(merged_df, "group", groups)

# Save the one_vs_all_results to a CSV
save_one_vs_all_results(one_vs_all_results_V4, '../Data/16S/random_forest/V4/one_vs_all_results_V4.csv')

Results saved to ../Data/16S/random_forest/V4/one_vs_all_results_V4.csv


In [ ]:
# Load the results CSV file
results_df = pd.read_csv('../Data/16S/random_forest/V4/one_vs_all_results_V4.csv')

# Specify the taxa of interest
taxa_of_interest = [
    'g__Staphylococcus','g__uncultured', 'g__Lawsonella', 'g__Streptococcus',
    'g__Acinetobacter', 'g__Cutibacterium', 'g__Enhydrobacter', 'g__Micrococcus', 'g__Lactobacillus', 'g__Rothia'
    #'g__Neisseria', 'g__Porphyromonas', 'g__Prevotella',  'g__Gemella', 'g__Veillonella',  'g__Haemophilus', 'g__Corynebacterium',
]

# Filter the data for the specified taxa
filtered_df = results_df[results_df['Feature'].str.strip().isin(taxa_of_interest)]

# Ensure the x-axis order is Healthy, Acne Non-lesional, Acne Lesional
group_order = ["Healthy", "Acne_NL", "Acne_L"]
filtered_df["Group"] = pd.Categorical(filtered_df["Group"], categories=group_order, ordered=True)

# Define custom colors for the taxa
taxa_colors = {
    'g__Cutibacterium': '#ffa505',  # Bright orange
    'g__Staphylococcus': '#92f0f0',  # Fluorescent light blue
    'g__Streptococcus': '#e2b46c',  # Beige
    'g__Corynebacterium': '#ffe59a',  # Pastel yellow
    'g__Lawsonella': '#70a8dc',  # Light blue
    'g__Veillonella': '#c5bce0',  # Pastel purplish
    'g__Micrococcus': '#f4cccd',  # Pastel yellow
    'g__Alloprevotella': '#bcbcbc',  # Light gray
    'g__Lactobacillus': '#daead3',  # Pale mint green
    'g__Rothia': '#ccb974', # Muted olive green
    'g__uncultured': '#f6475f',  # Pinkish
    'g__Neisseria': '#f6475f',  # Pinkish
    'g__Porphyromonas': '#64b5cd', # Light sky blue
    'g__Prevotella': '#4c72b0', # Muted royal blue 
    'g__Gemella': '#dd8452', # Light brown
    'g__Acinetobacter': '#55a868', # Evergreen
    'g__Haemophilus': '#c44e52', # Redish purple
    'g__Enhydrobacter': '#88748f' # Dark purple

}

# Strip leading/trailing spaces from 'Feature' column for consistency
filtered_df['Feature'] = filtered_df['Feature'].str.strip()

# Match the palette with the features in the filtered DataFrame
custom_palette = {feature: taxa_colors[feature] for feature in filtered_df['Feature'].unique()}

# Create the line plot
plt.figure(figsize=(7, 8))
sns.lineplot(
    data=filtered_df,
    x="Group",
    y="Importance",
    hue="Feature",
    marker="o",
    palette=custom_palette,
    hue_order=taxa_of_interest  # Explicitly set the order of taxa in the legend
)

# Add labels and title
plt.title("RF Driving Taxa by 16S V4", fontsize=20)
plt.xlabel("")
plt.ylabel("Feature Importance (Gini)", fontsize=18)
plt.xticks(ticks=[0, 1, 2], labels=["H", "ANL", "AL"], fontsize=18)
plt.yticks(fontsize=14)

# Create custom legend with circles in the middle of the lines
legend_elements = [
    Line2D(
        [0], [0],
        color=taxa_colors[taxon], marker='o', markersize=5, label=taxon,
        linestyle='-', linewidth=2
    )
    for taxon in taxa_of_interest if taxon in custom_palette
]

# Add the custom legend
plt.legend(
    handles=legend_elements,
    title="Top Genera",
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    title_fontsize=14,
    fontsize=14
)

# Save and show the plot
plt.tight_layout()
plt.savefig('../Figures/16S_Figures/primer_comparison/V4_Gini_importance.png', dpi=600)


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_76434/978125506.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["Group"] = pd.Categorical(filtered_df["Group"], categories=group_order, ordered=True)
/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_76434/978125506.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Feature'] = filtered_df['Feature'].str.strip()
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_ol